# 04. 리밸런싱 조정안 계산

이 Notebook에서 하는 일:
1. 현재 vs 목표 비중 비교
2. 매수 / 매도 금액 자동 산출
3. 세금 효율적 리밸런싱 순서 제안
4. 자산별 구체적 실행 계획
5. 실행 체크리스트 출력
6. 조정안 DB 저장

In [1]:
# 공통 설정
import pandas as pd
import numpy as np
import yaml, sqlite3, os
from pathlib import Path
from dotenv import load_dotenv
from datetime import date

load_dotenv()
PROJECT_ROOT = Path.cwd()

with open(PROJECT_ROOT / 'config.yaml', encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

MONTHLY_EXPENSE = CONFIG['user']['monthly_expense']
TARGET          = CONFIG['portfolio']
THRESHOLD       = TARGET['rebalance_threshold']  # 0.10

print(f"사용자: {CONFIG['user']['name']}")
print(f"리밸런싱 기준: ±{THRESHOLD*100:.0f}%p 초과 시 조정")

사용자: 종헌
리밸런싱 기준: ±10%p 초과 시 조정


In [2]:
# 자산 데이터 로드
assets_path = PROJECT_ROOT / 'assets.csv'
sample_path = PROJECT_ROOT / 'assets_sample.csv'

if assets_path.exists():
    df = pd.read_csv(assets_path, encoding='utf-8-sig')
else:
    df = pd.read_csv(sample_path, encoding='utf-8-sig')
    print('※ 샘플 데이터 사용 중')

mask = df['current_value'].isna() | (df['current_value'] == 0)
df.loc[mask, 'current_value'] = df.loc[mask, 'quantity'] * df.loc[mask, 'unit_price']
df['current_value'] = pd.to_numeric(df['current_value'], errors='coerce').fillna(0)

total = df['current_value'].sum()
type_sum = df.groupby('asset_type')['current_value'].sum()

current = {
    'cash':   type_sum.get('cash', 0),
    'bond':   type_sum.get('bond', 0) + type_sum.get('tdf', 0),
    'equity': type_sum.get('equity', 0),
    'income': type_sum.get('income', 0),
}

print(f'총 자산: {total:,.0f}원')
for k, v in current.items():
    print(f'  {k:8s}: {v:>15,.0f}원 ({v/total*100:.1f}%)')

※ 샘플 데이터 사용 중
총 자산: 67,550,000원
  cash    :      35,000,000원 (51.8%)
  bond    :      31,130,000원 (46.1%)
  equity  :       1,020,000원 (1.5%)
  income  :         400,000원 (0.6%)


## 1. 현재 vs 목표 비중 비교

In [3]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
pio.renderers.default = 'iframe'

labels_map = {'cash': '현금성', 'bond': '채권/TDF', 'equity': '주식형', 'income': '리츠/인컴'}
targets    = {'cash': TARGET['target_cash'], 'bond': TARGET['target_bond'],
              'equity': TARGET['target_equity'], 'income': TARGET['target_income']}

labels      = list(labels_map.values())
cur_ratios  = [current[k] / total for k in targets]
tgt_ratios  = [targets[k] for k in targets]
diffs       = [c - t for c, t in zip(cur_ratios, tgt_ratios)]

# 색상: 이탈 여부
bar_colors = ['#FF4444' if abs(d) >= THRESHOLD else '#70AD47' for d in diffs]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['목표 vs 현재 비중', '목표 대비 이탈(%p)'],
    specs=[[{'type': 'bar'}, {'type': 'bar'}]])

fig.add_trace(go.Bar(name='목표', x=labels, y=[t*100 for t in tgt_ratios],
    marker_color='#1F3864', text=[f'{t*100:.0f}%' for t in tgt_ratios], textposition='outside'),
    row=1, col=1)
fig.add_trace(go.Bar(name='현재', x=labels, y=[c*100 for c in cur_ratios],
    marker_color='#2E75B6', text=[f'{c*100:.1f}%' for c in cur_ratios], textposition='outside'),
    row=1, col=1)
fig.add_trace(go.Bar(name='이탈(%p)', x=labels, y=[d*100 for d in diffs],
    marker_color=bar_colors, text=[f'{d*100:+.1f}%p' for d in diffs], textposition='outside'),
    row=1, col=2)

fig.add_hline(y=THRESHOLD*100,  line_dash='dash', line_color='orange',
    annotation_text=f'+{THRESHOLD*100:.0f}%p 기준', row=1, col=2)
fig.add_hline(y=-THRESHOLD*100, line_dash='dash', line_color='orange',
    annotation_text=f'-{THRESHOLD*100:.0f}%p 기준', row=1, col=2)

fig.update_layout(title_text=f'리밸런싱 분석  |  총 자산 {total:,.0f}원',
    title_font_size=16, height=430, font=dict(size=13))
fig.show()

## 2. 매수 / 매도 금액 산출

In [4]:
print('=== 리밸런싱 조정 금액 ===')
print(f'총 자산: {total:,.0f}원')
print()

adjustments = {}
needs_rebalance = False

for k in targets:
    label   = labels_map[k]
    cur_amt = current[k]
    tgt_amt = total * targets[k]
    diff_amt = tgt_amt - cur_amt
    diff_pct = cur_amt / total - targets[k]
    adjustments[k] = {'label': label, 'current': cur_amt, 'target': tgt_amt,
                      'diff_amt': diff_amt, 'diff_pct': diff_pct}

    if abs(diff_pct) >= THRESHOLD:
        needs_rebalance = True
        action = '매수' if diff_amt > 0 else '매도'
        print(f'⚠️  {label:8s}: {action} {abs(diff_amt):>12,.0f}원  '
              f'(현재 {cur_amt/total*100:.1f}% → 목표 {targets[k]*100:.0f}%)')
    else:
        print(f'✅  {label:8s}: 조정 불필요  '
              f'(현재 {cur_amt/total*100:.1f}%, 목표 {targets[k]*100:.0f}%, '
              f'이탈 {diff_pct*100:+.1f}%p)')

print()
if not needs_rebalance:
    print('모든 자산군이 목표 범위 안에 있습니다. 리밸런싱이 필요하지 않습니다.')
else:
    total_trade = sum(abs(a['diff_amt']) for a in adjustments.values() if abs(a['diff_pct']) >= THRESHOLD) / 2
    print(f'예상 거래 규모: 약 {total_trade:,.0f}원 (매도 + 매수 합산의 절반)')

=== 리밸런싱 조정 금액 ===
총 자산: 67,550,000원

⚠️  현금성     : 매도   18,112,500원  (현재 51.8% → 목표 25%)
⚠️  채권/TDF  : 매도   14,242,500원  (현재 46.1% → 목표 25%)
⚠️  주식형     : 매수   22,622,500원  (현재 1.5% → 목표 35%)
⚠️  리츠/인컴   : 매수    9,732,500원  (현재 0.6% → 목표 15%)

예상 거래 규모: 약 32,355,000원 (매도 + 매수 합산의 절반)


## 3. 세금 효율적 리밸런싱 순서

리밸런싱 시 세금 부담을 줄이기 위한 우선순위:

1. **연금저축 / IRP 계좌 내** 리밸런싱 → 매도 시 세금 없음 (과세이연)
2. **신규 자금 투입** → 부족한 자산군 직접 매수 (매도 불필요)
3. **일반 계좌** 리밸런싱 → 매도 시 양도소득세 고려

In [5]:
# 계좌별 자산 분류
pension_accounts = ['연금저축', '연금', 'IRP', 'irp', '퇴직연금']

def is_pension(account_name):
    return any(kw in str(account_name) for kw in pension_accounts)

df['is_pension'] = df['account_name'].apply(is_pension)

pension_df = df[df['is_pension']]
general_df = df[~df['is_pension']]

pension_total = pension_df['current_value'].sum()
general_total = general_df['current_value'].sum()

print('=== 계좌별 자산 현황 ===')
print(f'연금계좌 (연금저축·IRP): {pension_total:>15,.0f}원 ({pension_total/total*100:.1f}%)')
print(f'일반계좌 (증권·예금 등):  {general_total:>15,.0f}원 ({general_total/total*100:.1f}%)')
print()
print('=== 세금 효율적 리밸런싱 순서 ===')
print()
print('1단계: 연금저축·IRP 계좌 내 자산 조정 (세금 없음)')
if not pension_df.empty:
    for _, row in pension_df.iterrows():
        print(f'   · {row["account_name"]} — {row["asset_name"]} ({row["current_value"]:,.0f}원)')
else:
    print('   (연금계좌 없음)')

print()
print('2단계: 신규 자금으로 부족한 자산군 직접 매수')
for k, a in adjustments.items():
    if a['diff_amt'] > 0 and abs(a['diff_pct']) >= THRESHOLD:
        print(f'   · {a["label"]}: {a["diff_amt"]:,.0f}원 매수 필요')

print()
print('3단계: 일반계좌 자산 매도 (양도소득세 확인 후 진행)')
if not general_df.empty:
    for _, row in general_df.iterrows():
        print(f'   · {row["account_name"]} — {row["asset_name"]} ({row["current_value"]:,.0f}원)')
else:
    print('   (일반계좌 없음)')

=== 계좌별 자산 현황 ===
연금계좌 (연금저축·IRP):      32,150,000원 (47.6%)
일반계좌 (증권·예금 등):       35,400,000원 (52.4%)

=== 세금 효율적 리밸런싱 순서 ===

1단계: 연금저축·IRP 계좌 내 자산 조정 (세금 없음)
   · 연금저축(예시) — TIGER 미국S&P500 (1,020,000원)
   · 연금저축(예시) — KODEX 단기채권 (10,250,000원)
   · IRP(예시) — TDF2035(미래에셋) (480,000원)
   · IRP(예시) — KODEX 국고채3년 (20,400,000원)

2단계: 신규 자금으로 부족한 자산군 직접 매수
   · 주식형: 22,622,500원 매수 필요
   · 리츠/인컴: 9,732,500원 매수 필요

3단계: 일반계좌 자산 매도 (양도소득세 확인 후 진행)
   · 예금(예시) — 정기예금(KB) (30,000,000원)
   · 예금(예시) — CMA(삼성) (5,000,000원)
   · 증권계좌(예시) — TIGER 리츠부동산인프라 (400,000원)


## 4. 자산별 구체적 실행 계획

In [6]:
print('=== 자산별 실행 계획 ===')
print()

action_items = []

for k, a in adjustments.items():
    if abs(a['diff_pct']) < THRESHOLD:
        continue

    label    = a['label']
    diff_amt = a['diff_amt']
    action   = '매수' if diff_amt > 0 else '매도'

    # 해당 자산군의 개별 자산 목록
    if k == 'bond':
        asset_rows = df[df['asset_type'].isin(['bond', 'tdf'])]
    else:
        asset_rows = df[df['asset_type'] == k]

    print(f'[{label}] → {action} {abs(diff_amt):,.0f}원')

    if asset_rows.empty:
        print(f'  · 보유 자산 없음 — 신규 매수 필요')
        hint = {'현금성': 'CMA 또는 파킹통장 입금',
                '채권/TDF': 'KODEX 단기채권 또는 TDF 매수',
                '주식형': 'TIGER 미국S&P500 또는 KODEX 200 매수',
                '리츠/인컴': 'TIGER 리츠부동산인프라 또는 배당ETF 매수'}.get(label, '')
        if hint:
            print(f'  · 추천: {hint}')
    else:
        asset_total = asset_rows['current_value'].sum()
        for _, row in asset_rows.iterrows():
            weight     = row['current_value'] / asset_total if asset_total > 0 else 0
            trade_amt  = abs(diff_amt) * weight
            if pd.notna(row.get('unit_price')) and row['unit_price'] > 0:
                units = trade_amt / row['unit_price']
                print(f'  · {row["asset_name"]:20s}: {action} {trade_amt:>12,.0f}원'
                      f'  (약 {units:.0f}주)')
            else:
                print(f'  · {row["asset_name"]:20s}: {action} {trade_amt:>12,.0f}원')
            action_items.append({'자산명': row['asset_name'], '계좌': row['account_name'],
                                 '행동': action, '금액': int(trade_amt)})
    print()

if not action_items and not needs_rebalance:
    print('현재 리밸런싱이 필요한 자산군이 없습니다.')

=== 자산별 실행 계획 ===

[현금성] → 매도 18,112,500원
  · 정기예금(KB)            : 매도   15,525,000원
  · CMA(삼성)             : 매도    2,587,500원

[채권/TDF] → 매도 14,242,500원
  · KODEX 단기채권          : 매도    4,689,548원  (약 46주)
  · TDF2035(미래에셋)       : 매도      219,608원  (약 14주)
  · KODEX 국고채3년         : 매도    9,333,344원  (약 92주)

[주식형] → 매수 22,622,500원
  · TIGER 미국S&P500      : 매수   22,622,500원  (약 1223주)

[리츠/인컴] → 매수 9,732,500원
  · TIGER 리츠부동산인프라      : 매수    9,732,500원  (약 2028주)



## 5. 실행 체크리스트

In [7]:
today_str = date.today().strftime('%Y년 %m월 %d일')

print(f'=== 리밸런싱 실행 체크리스트 ({today_str}) ===')
print()

checklist = [
    '현재 시장 상황 확인 (급락·급등 시 1~2주 대기 고려)',
    '연금저축·IRP 계좌 로그인 및 잔고 확인',
    '연금계좌 내 리밸런싱 먼저 실행 (세금 없음)',
    '일반계좌 매도 전 양도소득세 예상액 확인',
    '매도 → 2영업일 후 결제 확인',
    '매수 실행 (지정가 또는 시장가)',
    '01_data_input.ipynb 재실행하여 변경 후 현황 확인',
    '03_risk_score.ipynb 재실행하여 위험 점수 변화 확인',
]

for i, item in enumerate(checklist, 1):
    print(f'  [ ] {i}. {item}')

print()
print('※ 리밸런싱 완료 후 assets.csv를 업데이트하고 01_data_input.ipynb를 다시 실행하세요.')

if action_items:
    print()
    print('=== 거래 요약 ===')
    action_df = pd.DataFrame(action_items)
    print(action_df.to_string(index=False))

=== 리밸런싱 실행 체크리스트 (2026년 05월 20일) ===

  [ ] 1. 현재 시장 상황 확인 (급락·급등 시 1~2주 대기 고려)
  [ ] 2. 연금저축·IRP 계좌 로그인 및 잔고 확인
  [ ] 3. 연금계좌 내 리밸런싱 먼저 실행 (세금 없음)
  [ ] 4. 일반계좌 매도 전 양도소득세 예상액 확인
  [ ] 5. 매도 → 2영업일 후 결제 확인
  [ ] 6. 매수 실행 (지정가 또는 시장가)
  [ ] 7. 01_data_input.ipynb 재실행하여 변경 후 현황 확인
  [ ] 8. 03_risk_score.ipynb 재실행하여 위험 점수 변화 확인

※ 리밸런싱 완료 후 assets.csv를 업데이트하고 01_data_input.ipynb를 다시 실행하세요.

=== 거래 요약 ===
           자산명       계좌 행동       금액
      정기예금(KB)   예금(예시) 매도 15525000
       CMA(삼성)   예금(예시) 매도  2587500
    KODEX 단기채권 연금저축(예시) 매도  4689547
 TDF2035(미래에셋)  IRP(예시) 매도   219608
   KODEX 국고채3년  IRP(예시) 매도  9333344
TIGER 미국S&P500 연금저축(예시) 매수 22622500
TIGER 리츠부동산인프라 증권계좌(예시) 매수  9732500


## 6. 조정안 DB 저장

In [8]:
today   = str(date.today())
db_path = PROJECT_ROOT / 'portfolio.db'
conn    = sqlite3.connect(db_path)
cur     = conn.cursor()

saved = 0
for k, a in adjustments.items():
    if abs(a['diff_pct']) < THRESHOLD:
        continue
    action  = '매수' if a['diff_amt'] > 0 else '매도'
    message = (f"{a['label']} {action} {abs(a['diff_amt']):,.0f}원 "
               f"(현재 {a['current']/total*100:.1f}% → 목표 {targets[k]*100:.0f}%)")
    cur.execute('''
        INSERT INTO recommendations (date, rule_id, message, status)
        VALUES (?, ?, ?, ?)
    ''', (today, f'rebalance_{k}', message, 'pending'))
    saved += 1

conn.commit()
conn.close()

if saved:
    print(f'✅ 리밸런싱 조정안 {saved}건 저장 완료 ({today})')
else:
    print('저장할 조정안이 없습니다 (리밸런싱 불필요).')

✅ 리밸런싱 조정안 4건 저장 완료 (2026-05-20)


---

## 완료!

- 매수·매도 금액과 실행 체크리스트가 출력되었습니다.
- 리밸런싱 실행 후 `assets.csv`를 업데이트하고 `01_data_input.ipynb`부터 다시 실행하세요.

**다음 단계**: `05_llm_summary.ipynb` — GPT-4o-mini AI 3줄 요약 생성